<a href="https://colab.research.google.com/github/Thiva02/Statistical-Learning-e22399/blob/main/Assignment_4_Data_Wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comprehensive Data Preprocessing & Analytics Engine
A modular Python framework designed for end-to-end data sanitization, feature engineering, and interactive visual exploration in Google Colab.

## Core Capabilities

- **Smart Ingestion**: Interactively upload CSVs while automatically stripping out junk text (e.g., '?', 'N/A') and inferring proper numeric data types.
- **Data Auditing**: Generate instant structural summaries, highlighting shape, missing data, and type distributions.
- **Robust Cleansing**:
  - Flexible imputation (mean, median, mode, constant) for missing values.
  - IQR-based targeted outlier detection (flag or drop).
  - Exact duplicate removal and interactive column pruning.
- **Machine Learning Preparation**:
  - Scale continuous variables via Min-Max, Z-score (Standard), or Robust methods.
  - Encode categorical features via One-Hot, Ordinal, or Uniform (0-1 scaled) techniques.
- **Dynamic Visualizations (Plotly)**:
  - 3-panel univariate numeric charts (Horizontal Box, Scatter, Histogram).
  - Smart bivariate charts that adapt to data types (Scatter + OLS, Grouped Bar, or Box Plots).
- **Statistical Mapping**:
  - A unified correlation matrix that intelligently calculates Pearson’s $r$ (Numeric-Numeric), Cramér’s V (Category-Category), and Eta / Correlation Ratio (Mixed).

In [12]:
import pandas as pd
import numpy as np
import io
import scipy.stats as stats
from google.colab import files
from IPython.display import HTML, display
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OrdinalEncoder

class DataWrangler:
    """An object-oriented engine for automated data cleaning and ML feature preparation."""
    def __init__(self, dataset=None):
        self.df = dataset

    def import_csv_interactively(self):
        print("Awaiting CSV upload...")
        uploaded = files.upload()
        if not uploaded:
            return None
        file_key = list(uploaded.keys())[0]
        self.df = pd.read_csv(io.BytesIO(uploaded[file_key]))
        print(f"Data successfully loaded from: {file_key}")
        self._clean_null_placeholders()
        self._infer_and_convert_dtypes()
        return self.df

    def _clean_null_placeholders(self):
        junk_values = ['?', 'n/a', 'NULL', ' ', '', 'NA', 'N/A', '-']
        self.df.replace(junk_values, np.nan, inplace=True)

    def _infer_and_convert_dtypes(self):
        for feature in self.df.columns:
            converted_col = pd.to_numeric(self.df[feature], errors='coerce')
            if not converted_col.isna().all() or self.df[feature].isna().all():
                 self.df[feature] = pd.to_numeric(self.df[feature], errors='ignore')

    def display_dataset_info(self):
        """Displays dimensions, num/cat breakdown, and previews the first 20 rows."""
        if self.df is None:
            print("No data loaded.")
            return

        num_cols = len(self.df.select_dtypes(include=[np.number]).columns)
        cat_cols = len(self.df.select_dtypes(exclude=[np.number]).columns)

        print(f"Dataset Dimensions: {self.df.shape[0]} Observations, {self.df.shape[1]} Features")
        print(f"Feature Breakdown: {num_cols} Numerical, {cat_cols} Categorical\n")
        print("--- Detailed Feature Types ---")
        print(self.df.dtypes.value_counts())
        print("\n--- Data Preview (Top 20 Rows) ---")
        display(self.df.head(20))

    def impute_missing_data(self, method='mean', custom_value=None, target_cols=None):
        features = target_cols if target_cols else self.df.columns
        for col in features:
            if self.df[col].isnull().sum() == 0: continue
            if method == 'drop': self.df.dropna(subset=[col], inplace=True)
            elif method == 'constant':
                self.df[col] = self.df[col].fillna(custom_value)
            elif method == 'mean' and pd.api.types.is_numeric_dtype(self.df[col]):
                self.df[col] = self.df[col].fillna(self.df[col].mean())
            elif method == 'median' and pd.api.types.is_numeric_dtype(self.df[col]):
                self.df[col] = self.df[col].fillna(self.df[col].median())
            elif method == 'mode':
                self.df[col] = self.df[col].fillna(self.df[col].mode()[0])
        print(f"Null values resolved utilizing the '{method}' method.")

    def drop_exact_duplicates(self):
        starting_count = self.df.shape[0]
        self.df.drop_duplicates(inplace=True)
        print(f"Scrubbed {starting_count - self.df.shape[0]} duplicate records.")

    def manage_outliers_iqr(self, target_cols=None, treatment='flag'):
        features = target_cols if target_cols else self.df.select_dtypes(include=[np.number]).columns
        for col in features:
            q_low, q_high = self.df[col].quantile(0.25), self.df[col].quantile(0.75)
            iqr_val = q_high - q_low
            mask = (self.df[col] < q_low - 1.5 * iqr_val) | (self.df[col] > q_high + 1.5 * iqr_val)
            if treatment == 'drop': self.df = self.df[~mask]
            elif treatment == 'flag': self.df[f'{col}_outlier_flag'] = mask
        print(f"IQR Outlier detection complete. Treatment applied: '{treatment}'.")

    def remove_specified_columns(self, column_string):
        drop_list = [c.strip() for c in column_string.split(',')]
        existing = [c for c in drop_list if c in self.df.columns]
        self.df.drop(columns=existing, inplace=True)
        print(f"Successfully pruned features: {existing}")

    def remove_specified_rows(self, index_string):
        """Allows targeted deletion of rows based on comma-separated index values."""
        try:
            drop_list = [int(r.strip()) for r in index_string.split(',')]
            existing = [r for r in drop_list if r in self.df.index]
            self.df.drop(index=existing, inplace=True)
            print(f"Successfully pruned rows at indices: {existing}")
        except ValueError:
            print("Row deletion failed. Please ensure you provide comma-separated numbers (e.g., '0, 5, 12').")

    def scale_numerical_features(self, method='standard'):
        numerics = self.df.select_dtypes(include=[np.number]).columns
        if len(numerics) == 0: return pd.DataFrame()
        algorithm = MinMaxScaler() if method == 'minmax' else RobustScaler() if method == 'robust' else StandardScaler()
        transformed = algorithm.fit_transform(self.df[numerics])
        return pd.DataFrame(transformed, columns=numerics, index=self.df.index)

    def encode_categorical_features(self, method='onehot'):
        categoricals = self.df.select_dtypes(exclude=[np.number]).columns
        if len(categoricals) == 0: return pd.DataFrame()
        if method == 'onehot':
            return pd.get_dummies(self.df[categoricals], drop_first=True, dtype=int)
        ordinal_data = OrdinalEncoder().fit_transform(self.df[categoricals])
        encoded_frame = pd.DataFrame(ordinal_data, columns=categoricals, index=self.df.index)
        if method == 'uniform':
            encoded_frame = pd.DataFrame(MinMaxScaler().fit_transform(encoded_frame), columns=categoricals, index=self.df.index)
        return encoded_frame

    def combine_transformed_data(self, numeric_frame, categorical_frame):
        return pd.concat([numeric_frame, categorical_frame], axis=1)

class InsightVisualizer:
    """A dedicated class for rendering interactive Plotly charts and statistical heatmaps."""
    def __init__(self, dataset):
        self.dataset = dataset.copy()

    def _render_html(self, figure):
        """Helper method to ensure HTML rendering in Colab."""
        display(HTML(figure.to_html(full_html=False, include_plotlyjs='cdn')))

    # --- Core Required Visualizations ---

    def render_numeric_distribution(self, feature_name):
        """Generates a 3-panel subplot: Box Plot, Scatter Plot, and Histogram."""
        subset = self.dataset[feature_name].dropna()
        layout = make_subplots(rows=1, cols=3, subplot_titles=("Box Plot", "Sequential Scatter", "Histogram"))
        layout.add_trace(go.Box(x=subset, name=feature_name, orientation='h'), row=1, col=1)
        layout.add_trace(go.Scatter(x=subset.index, y=subset, mode='markers'), row=1, col=2)
        layout.add_trace(go.Histogram(x=subset), row=1, col=3)
        layout.update_layout(title_text=f"Distribution Profile: {feature_name}", showlegend=False, height=400)
        self._render_html(layout)

    def render_category_counts(self, feature_name):
        """Generates a bar chart with raw counts and percentage labels."""
        freq = self.dataset[feature_name].value_counts().reset_index()
        freq.columns = [feature_name, 'Total']
        freq['Ratio'] = (freq['Total'] / freq['Total'].sum() * 100).round(2).astype(str) + '%'
        chart = px.bar(freq, x=feature_name, y='Total', text='Ratio', title=f"Frequency Analysis: {feature_name}", color=feature_name)
        self._render_html(chart)

    def render_bivariate_analysis(self, x_feature, y_feature):
        """Intelligently detects datatypes and plots the appropriate relationship chart."""
        clean_df = self.dataset.dropna(subset=[x_feature, y_feature])
        x_is_num = pd.api.types.is_numeric_dtype(self.dataset[x_feature])
        y_is_num = pd.api.types.is_numeric_dtype(self.dataset[y_feature])

        if x_is_num and y_is_num:
            chart = px.scatter(clean_df, x=x_feature, y=y_feature, trendline="ols", title=f"Correlation: {x_feature} vs {y_feature}")
        elif not x_is_num and not y_is_num:
            tally = clean_df.groupby([x_feature, y_feature]).size().reset_index(name='Total')
            chart = px.bar(tally, x=x_feature, y='Total', color=y_feature, barmode='group', title=f"Cross-Tab: {x_feature} vs {y_feature}")
        else:
            cat_var, num_var = (x_feature, y_feature) if not x_is_num else (y_feature, x_feature)
            chart = px.box(clean_df, x=cat_var, y=num_var, points="all", title=f"Spread: {num_var} grouped by {cat_var}")
        self._render_html(chart)

    # --- Granular Custom Charts (Section 6 Requirement) ---

    def render_pie_chart(self, category_col, hole_size=0.4):
        """Standalone method for generating Pie/Donut charts."""
        chart = px.pie(self.dataset, names=category_col, hole=hole_size, title=f"Proportional Split: {category_col}")
        self._render_html(chart)

    def render_custom_bar(self, x_col, y_col, color_col=None, mode='group'):
        """Standalone method for generating granular Custom Bar charts."""
        chart = px.bar(self.dataset, x=x_col, y=y_col, color=color_col, barmode=mode, title=f"Bar Chart: {y_col} by {x_col}")
        self._render_html(chart)

    def render_custom_histogram(self, num_col, bins=None):
        """Standalone method for generating granular Histograms."""
        chart = px.histogram(self.dataset, x=num_col, nbins=bins, title=f"Custom Histogram: {num_col}")
        self._render_html(chart)

    # --- Advanced Statistical Methods ---

    def _calc_cramers_v(self, cat1, cat2):
        crosstab = pd.crosstab(cat1, cat2)
        chi_sq = stats.chi2_contingency(crosstab)[0]
        total_obs = crosstab.sum().sum()
        return np.sqrt((chi_sq/total_obs) / min(crosstab.shape[0]-1, crosstab.shape[1]-1)) if total_obs > 0 and min(crosstab.shape[0]-1, crosstab.shape[1]-1) > 0 else 0.0

    def _calc_correlation_ratio(self, categories, numbers):
        factored, _ = pd.factorize(categories)
        num_cats = np.max(factored) + 1
        avg_y, counts = np.zeros(num_cats), np.zeros(num_cats)
        for idx in range(num_cats):
            subset_nums = numbers[np.argwhere(factored == idx).flatten()]
            counts[idx] = len(subset_nums)
            avg_y[idx] = np.average(subset_nums) if len(subset_nums) > 0 else 0
        overall_avg = np.sum(avg_y * counts) / np.sum(counts)
        numerator = np.sum(counts * ((avg_y - overall_avg)**2))
        denominator = np.sum((numbers - overall_avg)**2)
        return np.sqrt(numerator / denominator) if denominator != 0 else 0.0

    def render_unified_correlation_heatmap(self):
        """Generates a matrix using Pearson's r, Cramer's V, and Correlation Ratio."""
        features = self.dataset.columns
        corr_grid = pd.DataFrame(np.ones((len(features), len(features))), index=features, columns=features)
        for i, f1 in enumerate(features):
            for j, f2 in enumerate(features):
                if i == j: continue
                pair_data = self.dataset[[f1, f2]].dropna()
                if pair_data.empty:
                    corr_grid.loc[f1, f2] = 0.0
                    continue
                is_n1, is_n2 = pd.api.types.is_numeric_dtype(self.dataset[f1]), pd.api.types.is_numeric_dtype(self.dataset[f2])
                if is_n1 and is_n2: corr_grid.loc[f1, f2] = pair_data[f1].corr(pair_data[f2])
                elif not is_n1 and not is_n2: corr_grid.loc[f1, f2] = self._calc_cramers_v(pair_data[f1], pair_data[f2])
                else: corr_grid.loc[f1, f2] = self._calc_correlation_ratio(pair_data[f1 if not is_n1 else f2], pair_data[f2 if not is_n1 else f1].values)
        chart = px.imshow(corr_grid.round(2), text_auto=True, color_continuous_scale='Tealrose', zmin=-1, zmax=1, title="Global Statistical Association Heatmap")
        self._render_html(chart)

## Phase 1: Data Ingestion

In [13]:
# Bootstrapping the Wrangler
wrangler = DataWrangler()

# Loading the Palmer Penguins dataset directly via URL for the demonstration
penguins_url = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/master/inst/extdata/penguins.csv"
wrangler.df = pd.read_csv(penguins_url)
print("Palmer Penguins dataset successfully ingested.")

Palmer Penguins dataset successfully ingested.


## Phase 2: Structural Auditing & Cleansing

In [14]:
# 1. Audit the raw data
wrangler.display_dataset_info()

# 2. Prune unnecessary administrative columns and specific row indices
wrangler.remove_specified_columns("year")
wrangler.remove_specified_rows("0, 1, 2") # Testing targeted row deletion

# 3. Handle missing biological data (Imputing numerical with median, categorical with mode)
wrangler.impute_missing_data(method='median', target_cols=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'])
wrangler.impute_missing_data(method='mode', target_cols=['sex'])

# 4. Scrub any anomalous extreme values from body mass
wrangler.manage_outliers_iqr(target_cols=['body_mass_g'], treatment='drop')
wrangler.drop_exact_duplicates()

Dataset Dimensions: 344 Observations, 8 Features
Feature Breakdown: 5 Numerical, 3 Categorical

--- Detailed Feature Types ---
float64    4
object     3
int64      1
Name: count, dtype: int64

--- Data Preview (Top 20 Rows) ---


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,male,2007
6,Adelie,Torgersen,38.9,17.8,181.0,3625.0,female,2007
7,Adelie,Torgersen,39.2,19.6,195.0,4675.0,male,2007
8,Adelie,Torgersen,34.1,18.1,193.0,3475.0,NaN,2007
9,Adelie,Torgersen,42.0,20.2,190.0,4250.0,NaN,2007


Successfully pruned features: ['year']
Successfully pruned rows at indices: [0, 1, 2]
Null values resolved utilizing the 'median' method.
Null values resolved utilizing the 'mode' method.
IQR Outlier detection complete. Treatment applied: 'drop'.
Scrubbed 0 duplicate records.


## Phase 3: Machine Learning Feature Preparation

In [15]:
# MinMax scale the penguin measurements
scaled_metrics = wrangler.scale_numerical_features(method='minmax')

# Apply Uniform (0-1) Ordinal encoding to the categories (Island, Species, Sex)
encoded_factors = wrangler.encode_categorical_features(method='uniform')

# Stitch them together into an ML-ready structure
model_ready_data = wrangler.combine_transformed_data(scaled_metrics, encoded_factors)
print("\n--- ML-Ready Dataset Snippet ---")
display(model_ready_data.head())


--- ML-Ready Dataset Snippet ---


,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,species,island,sex
3,0.450909,0.500000,0.423729,0.375000,0.0,1.0,1.0
4,0.167273,0.738095,0.355932,0.208333,0.0,1.0,0.0
5,0.261818,0.892857,0.305085,0.263889,0.0,1.0,1.0
6,0.247273,0.559524,0.152542,0.256944,0.0,1.0,0.0
7,0.258182,0.773810,0.389831,0.548611,0.0,1.0,1.0


## Phase 4: Advanced Visual Analytics

In [16]:
# Spin up the Visualizer using our cleaned (but unscaled) data for interpretable axes
visuals = InsightVisualizer(wrangler.df)

# 1. 3-Panel Univariate Analysis
visuals.render_numeric_distribution('flipper_length_mm')

# 2. Smart Bivariate Analysis
visuals.render_bivariate_analysis('island', 'body_mass_g')

# 3. Granular Custom Charts (Pie & Custom Bar)
visuals.render_pie_chart('island', hole_size=0.5)
visuals.render_custom_bar(x_col='species', y_col='body_mass_g', color_col='sex')

# 4. Global Association Heatmap
visuals.render_unified_correlation_heatmap()